# [센서 실습 26] UV 화염 감지 센서 – UVTron R2868 + C10807 구동 보드

불꽃의 **자외선**에만 반응하는 **UVTron** 센서가 내보내는 **펄스 수를 세어** 화재를 감지합니다. 햇빛·조명에는 반응하지 않는 '태양광 블라인드' 특성을 7번 적외선 센서와 비교합니다.

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
불꽃은 **185~260nm 파장의 자외선(UV-C)** 을 냅니다. 이 파장의 햇빛은 대기의 오존층에서 거의 흡수되어 땅에 닿지 않습니다.
- UVTron은 이 파장에만 반응하는 **방전관**: 자외선 광자가 전극에 닿으면 순간 방전이 일어나 **펄스** 1개 발생
- 구동 보드(C10807)가 고전압을 공급하고, 방전마다 **10ms 펄스**를 출력
- 불꽃이 클수록·가까울수록 **초당 펄스 수가 많아짐** → 펄스를 세어 판단
- 우주선(cosmic ray) 등으로 가끔 펄스가 생기므로 **일정 시간 동안 펄스 수**로 판정
- 햇빛·백열등·적외선에는 반응하지 않음 → **오경보가 적어** 공장·주유소 화재 감지기에 사용

### 부품 사양
| 항목 | 내용 |
|---|---|
| 감지 파장 | 185 ~ 260 nm |
| 감도 | 라이터 불꽃 5m 거리에서 감지 |
| 배경 펄스 | 약 10회/분 이하 |
| 출력 | 펄스 (방전 1회당 10ms LOW, 보드 설정) |
| 전원 | 구동 보드 5V |

## 2. 라즈베리파이 연결 방법
**준비물**: UVTron R2868 + C10807 구동 보드, 점퍼선 (구동 보드는 고전압을 만들므로 동작 중 만지지 않기)

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| C10807 +5V / GND | 5V (2번 핀) / GND |  |
| C10807 출력 (오픈 컬렉터) | GPIO27 (13번 핀) | 내부 풀업 사용, 펄스마다 LOW |



**인터페이스 활성화**
- **GPIO**: 별도 설정 없이 바로 사용 가능

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
pip3 install gpiozero
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
UVTron은 값을 주지 않고 **펄스만** 줍니다. 1초 동안 센 펄스 수로 판정하는 함수를 공통으로 씁니다.

In [ ]:
FLAME_CPS = 5          # 초당 펄스 5개 이상이면 불꽃 의심
CONFIRM_S = 2          # 2초 연속이면 화재 확정

def judge_flame(counts):
    """초당 펄스 수 목록 → 화재 확정 여부 목록"""
    run, out = 0, []
    for c in counts:
        run = run + 1 if c >= FLAME_CPS else 0
        out.append(int(run >= CONFIRM_S))
    return out

### 4-2. 실제 센서 클래스
GPIO 입력에 **콜백 함수**를 걸어 펄스가 올 때마다 개수를 늘리고, `read()`할 때 지난 1초 동안의 개수를 돌려줍니다. (계속 확인하는 대신 '사건이 생기면 알려 주는' 방식)

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealUVTron:
    def __init__(self, pin=27):
        from gpiozero import DigitalInputDevice
        self.dev = DigitalInputDevice(pin, pull_up=True)
        self.count = 0
        self.dev.when_activated = self._on_pulse          # 펄스(LOW)가 들어올 때마다 호출

    def _on_pulse(self):
        self.count += 1

    def read(self):
        c, self.count = self.count, 0                      # 지난 측정 이후 펄스 수
        return {"pulses": c}

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
| 시간 | 가정한 상황 | 초당 평균 펄스 |
|---|---|---|
| 항상 | 배경 (우주선 등) | 0.1 |
| 20 ~ 35초 | 창으로 **강한 햇빛**이 비침 | 0.1 (반응 없음) |
| 60 ~ 75초 | 3m 거리에서 **라이터 불꽃** | 15 |
| 90초 | 순간 전기 스파크 1회 | 1초만 8 |

펄스는 무작위로 생기므로 **포아송 분포**(일정한 평균으로 드문드문 생기는 사건의 개수)로 만들었습니다.

In [ ]:
class VirtualUVTron:
    def __init__(self, dt=1.0, seed=26):
        self.dt, self.t = dt, 0.0
        self.rng = np.random.default_rng(seed)

    def read(self):
        rate = 0.1                                          # 배경
        if 60 <= self.t < 75: rate = 15                     # 라이터 불꽃
        if self.t == 90: rate = 8                           # 순간 스파크
        sunlight = 20 <= self.t < 35                        # 햇빛: UVTron은 반응하지 않음
        c = int(self.rng.poisson(rate * self.dt))
        self.t += self.dt
        return {"pulses": c, "sunlight": int(sunlight)}

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualUVTron(dt=1.0)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
1초마다 펄스 수를 세어 2분 동안 기록합니다.

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 120               # 측정 횟수
INTERVAL = 1.0               # 측정 간격(초)

sensor = RealUVTron() if USE_REAL_SENSOR else VirtualUVTron(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
초당 펄스 **5개 이상이 2초 연속**이면 화재로 확정합니다. 순간 스파크(1초)는 걸러집니다.

In [ ]:
df["fire"] = judge_flame(df.pulses)
print("전체 펄스 수:", df.pulses.sum())
if df.fire.any():
    f = df[df.fire == 1]
    print(f"🔥 화재 확정: {f.time_s.min():.0f}초 ~ {f.time_s.max():.0f}초 → 경보·스프링클러 작동")
print("햇빛 구간 펄스:", df[df.get("sunlight", 0) == 1].pulses.sum() if "sunlight" in df else "-",
      "→ 햇빛에는 반응하지 않음 (7번 적외선 센서와 비교)")
print("90초 스파크:", int(df.loc[df.time_s == 90, "pulses"].iloc[0]), "펄스 → 1초뿐이라 화재로 판정하지 않음")

## 8. 시각화

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 5), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax[0].bar(df.time_s, df.pulses, width=0.9, color="gray", label="초당 펄스")
ax[0].axhline(FLAME_CPS, color="black", ls="--", label=f"불꽃 기준 {FLAME_CPS}/초")
if "sunlight" in df:
    ax[0].fill_between(df.time_s, 0, df.pulses.max(), where=df.sunlight == 1, color="0.93", label="햇빛 구간")
ax[0].set_ylabel("펄스 수"); ax[0].legend(); ax[0].set_title("UVTron 자외선 화염 감지")
ax[1].step(df.time_s, df.fire, "k-", where="post"); ax[1].fill_between(df.time_s, df.fire, step="post", color="lightgray")
ax[1].set_yticks([0, 1]); ax[1].set_yticklabels(["정상", "화재"]); ax[1].set_xlabel("시간 (초)")
plt.tight_layout(); plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "UV(화염감지)센서_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "UVTron R2868 + C10807 구동 보드",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/flame_uv", payload)
# client.disconnect()

## 직접 해 보기
- `CONFIRM_S`를 1로 바꾸면 90초 스파크가 어떻게 판정되나요?
- 불꽃이 멀어져 초당 펄스가 3개로 줄었다면 기준을 어떻게 정해야 할까요?
- 7번 적외선 센서와 UV 센서를 함께 써서 둘 다 감지할 때만 경보하는 방식의 장단점을 정리해 보세요.